In [5]:
# main_train_concat.py
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path
import matplotlib
matplotlib.use('Agg') 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import numpy as np

# ================= 配置区域 =================
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# [确保这里与数据生成器一致]
DATASET_ROOT = "./dataset_tri_RAD_Concat" 
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")
RESULTS_DIR = "./training_results_concat" 

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]
NUM_CLASSES = len(CLASS_NAMES)

# [拼接版尺寸配置]
IMAGE_HEIGHT = 64
IMAGE_WIDTH = 192 # 64 * 3
BATCH_SIZE = 128
LEARNING_RATE = 0.001
NUM_EPOCHS = 15

os.makedirs(RESULTS_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(RESULTS_DIR, "best_model.pth")

# ================= 1. 数据集定义 (灰度模式) =================
class InMemoryDataset(Dataset):
    def __init__(self, root_dir, has_snr=False):
        self.data = []
        self.labels = []
        self.snrs = [] 
        
        # 必须 Resize 到 (64, 192) 以匹配生成的数据
        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_HEIGHT, IMAGE_WIDTH)), 
            transforms.ToTensor(), 
        ])
        
        if not os.path.exists(root_dir):
            print(f"Path not found: {root_dir}")
            return
            
        print(f"Loading {root_dir} into RAM...")
        files = []
        for cls_name in CLASS_NAMES:
            d = Path(root_dir) / cls_name
            if d.exists(): 
                files.extend([(p, CLASS_NAMES.index(cls_name)) for p in d.glob("*.png")])
        
        with tqdm(total=len(files), unit="img") as pbar:
            for p, label in files:
                try:
                    snr = -999.0
                    if has_snr:
                        try:
                            snr = float(p.stem.split('_')[-1])
                        except: pass
                    
                    # [关键] 读取为灰度图 'L' (1通道)
                    with Image.open(str(p)).convert("L") as img:
                        tensor = self.transform(img)
                        # [黑底修正] 防止反色（确保背景是0，线条是亮色）
                        if tensor.mean() > 0.5:
                            tensor = 1.0 - tensor
                            
                        self.data.append(tensor) 
                        self.labels.append(label)
                        self.snrs.append(snr)
                except Exception:
                    pass
                pbar.update(1)

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): 
        return self.data[idx], self.labels[idx], self.snrs[idx]

# ================= 2. 模型定义 (1通道输入) =================
class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super(TinyCNN, self).__init__()
        self.features = nn.Sequential(
            # [关键] 输入通道=1 (灰度)
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Dropout2d(0.1),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2)
        )
        
        # 尺寸计算: 
        # (64, 192) -> /2 -> (32, 96) -> /2 -> (16, 48) -> /2 -> (8, 24)
        # Flatten: 64通道 * 8 * 24
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 24, 128), 
            nn.ReLU(), 
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

# ================= 3. 辅助绘图函数 =================
def plot_training_curves(history, save_path):
    epochs = range(1, len(history['train_acc']) + 1)
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-', label='Train')
    plt.plot(epochs, history['val_loss'], 'r-', label='Val') 
    plt.title('Loss')
    plt.legend()
    plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'b-', label='Train')
    plt.plot(epochs, history['val_acc'], 'r-', label='Val')
    plt.title('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path)
    plt.close()

def plot_confusion_matrix(y_true, y_pred, title, save_path, target_names=None, labels=None):
    """
    绘制混淆矩阵。
    如果提供了 labels (索引列表) 和 target_names (对应的名字列表)，
    则只会绘制这些指定的类别。
    """
    # 使用 labels 参数来筛选矩阵
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    # 如果没有指定，默认显示所有类别
    if target_names is None: 
        target_names = CLASS_NAMES
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=target_names, yticklabels=target_names)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

# ================= 4. 主训练逻辑 =================
def main():
    print(f"Initializing Concatenated Dataset ({IMAGE_HEIGHT}x{IMAGE_WIDTH})...")
    train_ds = InMemoryDataset(SIM_TRAIN_DIR, has_snr=False) 
    val_ds = InMemoryDataset(SIM_VAL_DIR, has_snr=True) 
    real_ds = InMemoryDataset(REAL_TEST_DIR, has_snr=False)
    
    if len(train_ds) == 0:
        print("No data! Run data_generator_concat.py first.")
        return

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    model = TinyCNN(NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    
    print(f"\nStarting Training for {NUM_EPOCHS} epochs...")
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        
        for imgs, labels, _ in train_loader: 
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                outputs = model(imgs)
                val_loss_sum += criterion(outputs, labels).item() * imgs.size(0)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)
        
        val_loss = val_loss_sum / val_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(epoch_loss)
        history['train_acc'].append(epoch_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"Epoch {epoch+1} | Train: {epoch_acc:.4f} | Val: {val_acc:.4f} [Saved Best]")
        else:
            print(f"Epoch {epoch+1} | Train: {epoch_acc:.4f} | Val: {val_acc:.4f}")

    plot_training_curves(history, os.path.join(RESULTS_DIR, "training_curves.png"))
    
    # --- 测试部分 ---
    print("\nLoading Best Model...")
    model.load_state_dict(torch.load(BEST_MODEL_PATH))
    model.eval()
    
    # 真实数据测试
    if len(real_ds) > 0:
        print("Testing Real Data...")
        real_loader = DataLoader(real_ds, batch_size=BATCH_SIZE)
        y_true, y_pred = [], []
        with torch.no_grad():
            for imgs, labels, _ in real_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                _, preds = model(imgs).max(1)
                y_true.extend(labels.cpu().numpy())
                y_pred.extend(preds.cpu().numpy())
        
        # 转换为 numpy 数组
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        
        acc = np.mean(y_true == y_pred)
        print(f"Real Data Accuracy: {acc:.4f}")
        
        # === [核心修改] 动态筛选存在的类别 ===
        # 1. 找出真实数据中存在的类别索引
        existing_indices = sorted(list(set(y_true)))
        
        # 2. 找出对应的类别名称
        existing_names = [CLASS_NAMES[i] for i in existing_indices]
        
        print(f"Classes present in Real Data: {existing_names}")
        
        # 3. 只绘制这些类别
        plot_confusion_matrix(
            y_true, 
            y_pred, 
            f"Real Data (Acc: {acc:.2%})", 
            os.path.join(RESULTS_DIR, "real_cm.png"),
            target_names=existing_names,  # 只传入存在的名字
            labels=existing_indices       # 只传入存在的索引
        )
        print(f"Filtered Confusion Matrix saved to {RESULTS_DIR}")
    else:
        print("No real data found for testing.")

if __name__ == "__main__":
    main()

Initializing Concatenated Dataset (64x192)...
Loading ./dataset_tri_RAD_Concat\sim_train into RAM...


100%|██████████| 7693/7693 [00:02<00:00, 3649.27img/s]


Loading ./dataset_tri_RAD_Concat\sim_val into RAM...


100%|██████████| 1907/1907 [00:00<00:00, 3725.93img/s]


Loading ./dataset_tri_RAD_Concat\real_test into RAM...


100%|██████████| 43938/43938 [00:11<00:00, 3858.74img/s]



Starting Training for 15 epochs...
Epoch 1 | Train: 0.1716 | Val: 0.4504 [Saved Best]
Epoch 2 | Train: 0.3029 | Val: 0.5931 [Saved Best]
Epoch 3 | Train: 0.3537 | Val: 0.6120 [Saved Best]
Epoch 4 | Train: 0.3919 | Val: 0.6959 [Saved Best]
Epoch 5 | Train: 0.4323 | Val: 0.7902 [Saved Best]
Epoch 6 | Train: 0.4560 | Val: 0.7771
Epoch 7 | Train: 0.4717 | Val: 0.7179
Epoch 8 | Train: 0.4773 | Val: 0.8123 [Saved Best]
Epoch 9 | Train: 0.5105 | Val: 0.8217 [Saved Best]
Epoch 10 | Train: 0.5275 | Val: 0.8028
Epoch 11 | Train: 0.5525 | Val: 0.8086
Epoch 12 | Train: 0.5560 | Val: 0.7514
Epoch 13 | Train: 0.5725 | Val: 0.8474 [Saved Best]
Epoch 14 | Train: 0.5861 | Val: 0.8437
Epoch 15 | Train: 0.6047 | Val: 0.8201

Loading Best Model...
Testing Real Data...
Real Data Accuracy: 0.6688
Classes present in Real Data: ['BPSK', 'QPSK', '8PSK', 'OQPSK', 'PI4DQPSK', '16QAM', '64QAM', '2ASK', 'FSK2']
Filtered Confusion Matrix saved to ./training_results_concat
